# lib_coreII
## Núcleo de análisis y síntesis de plantas y controladores

**Línea evolutiva:** `lib_basicaControlClasico` → `lib_coreControlClasico` → **`lib_coreII`**

Este cuaderno es a la vez tres cosas:

1. **Un libro técnico** sobre cómo se construye una herramienta de control: de dónde viene la robustez numérica de MATLAB/Octave, qué ofrece el ecosistema Python y qué decisiones de diseño hay que tomar.
2. **Una librería ejecutable** (`lib_coreII`), documentada estilo Doxygen, validada contra resultados analíticos de la teoría.
3. **Un banco de trabajo** con ejemplos progresivos de análisis y síntesis: respuesta temporal, Bode, Nyquist, lugar de raíces, Routh–Hurwitz, compensadores de adelanto/atraso (con diseño automático por margen de fase), PID realizable, retardo de Padé, discretización, ubicación de polos, LQR y observadores.

---

## Índice

**Parte I — Fundamentos y contexto**
1. Prólogo: del álgebra a la herramienta
2. Contexto histórico: Fortran, BLAS, LAPACK y el origen de MATLAB
3. El ecosistema Python de control: madurez real de las alternativas
4. Filosofía de diseño de `lib_coreII`
5. Entorno y dependencias

**Parte II — El núcleo (código documentado)**
6. Utilidades numéricas
7. Clase `TF`: función de transferencia (continua y discreta)
8. Clase `SS`: espacio de estados
9. Interconexiones: serie, paralelo y realimentación
10. Métricas temporales: `step_info`
11. Márgenes de estabilidad robustos
12. Gráficos: escalón, impulso, Bode, Nyquist
13. Lugar geométrico de las raíces con ramas continuas
14. Routh–Hurwitz
15. Discretización y retardo de Padé
16. Síntesis clásica: lead, lag, lead–lag, PID y diseño automático
17. Síntesis moderna: ubicación de polos, LQR y observadores
18. Modelos rápidos y reportes

**Parte III — Validación**
19. Suite de autotests contra resultados analíticos
20. Validación cruzada opcional contra `python-control`

**Parte IV — Aplicación**
21. Ejemplo 1: primer orden
22. Ejemplo 2: segundo orden subamortiguado
23. Ejemplo 3: lazo cerrado, Nyquist y lugar de raíces
24. Ejemplo 4: Routh–Hurwitz y rango de ganancias estables
25. Ejemplo 5: diseño automático de adelanto por margen de fase
26. Ejemplo 6: atraso para error estacionario
27. Ejemplo 7: retardo de transporte con Padé
28. Ejemplo 8: discretización ZOH vs. Tustin
29. Ejemplo 9: espacio de estados, LQR y observador
30. Criterios prácticos de ingeniería
31. Hoja de ruta
32. Referencias


# 1. Prólogo: del álgebra a la herramienta

En control automático trabajamos con objetos matemáticos precisos. Para un sistema lineal e invariante en el tiempo (LTI), la relación entrada–salida es una convolución:

$$
y(t) = g(t) * x(t)
$$

y la transformada de Laplace la convierte en producto:

$$
Y(s) = G(s)\,X(s), \qquad G(s)=\frac{N(s)}{D(s)}
$$

De ahí nace toda el álgebra de bloques: la cascada multiplica, el paralelo suma, y la realimentación negativa unitaria produce

$$
T(s)=\frac{G(s)}{1+G(s)}
$$

Hasta aquí, matemática. El problema aparece cuando queremos **calcular**: raíces de polinomios de grado alto, autovalores, ecuaciones de Lyapunov y de Riccati, barridos de frecuencia con miles de puntos. Ese salto —del álgebra al número en coma flotante— es donde una herramienta se gana o pierde su robustez. Y es exactamente el salto que MATLAB y Octave resolvieron muy bien hace décadas, no por magia, sino apoyándose en una infraestructura numérica que vale la pena conocer.


# 2. Contexto histórico: Fortran, BLAS, LAPACK y el origen de MATLAB

La "robustez de MATLAB" tiene nombre y apellido, y es más vieja que MATLAB.

## 2.1 Fortran y las librerías fundacionales

- **1957 — Fortran** (FORmula TRANslation, IBM): el primer lenguaje de alto nivel pensado para cálculo científico. Durante medio siglo fue *el* lenguaje del análisis numérico, y su semántica de arrays sigue siendo ideal para que los compiladores optimicen álgebra lineal.
- **Años 70 — EISPACK y LINPACK** (Argonne National Laboratory): colecciones de rutinas Fortran para autovalores (EISPACK) y sistemas lineales/factorizaciones (LINPACK). Décadas de análisis de error, escalado, balanceo y pivoteo destiladas en código.
- **1979 — BLAS** (Basic Linear Algebra Subprograms): estandariza las operaciones básicas (producto matriz–vector, matriz–matriz) para que el código portable pueda apoyarse en implementaciones optimizadas por arquitectura.
- **1992 — LAPACK**: sucesor de EISPACK + LINPACK, reescrito sobre BLAS de nivel 3 para aprovechar jerarquías de memoria. Hoy sigue siendo el estándar de facto: factorizaciones QR, LU, Cholesky, SVD, autovalores (rutinas como `dgeev`), ecuaciones de Sylvester/Lyapunov.

## 2.2 MATLAB nace como envoltorio

A fines de los 70, **Cleve Moler** —coautor de EISPACK y LINPACK— quería que sus estudiantes usaran esas rutinas **sin programar en Fortran**. Escribió un intérprete interactivo llamado *MATrix LABoratory*: MATLAB clásico era, literalmente, una interfaz de comandos sobre EISPACK/LINPACK. En 1984 se funda MathWorks y el producto se reescribe en C, pero el corazón numérico siguió (y sigue) delegando en BLAS/LAPACK optimizados.

Para control en particular existe además **SLICOT** (Subroutine Library In COntrol Theory): una librería Fortran 77 con algoritmos especializados y numéricamente cuidados para Riccati, Lyapunov, reducción de modelos, formas de Schur, etc. Tanto MATLAB como Octave y `python-control` (vía `slycot`) se apoyan o se inspiran en ella para los problemas difíciles.

## 2.3 La conclusión histórica importante

> Cuando NumPy resuelve `np.linalg.eigvals(A)`, está llamando a la **misma familia de rutinas LAPACK** que usa MATLAB. La robustez de base es compartida; lo que difiere es la **capa intermedia**: cuánto cuidado pone cada herramienta en normalizar, escalar, elegir representaciones y manejar casos patológicos *antes* de llamar al núcleo compilado.

Por eso este cuaderno no intenta "reescribir LAPACK en Python" (sería un retroceso), sino construir una **capa intermedia cuidadosa** sobre NumPy/SciPy: ahí es donde una librería casera suele perder robustez, y ahí es donde podemos ganarla.


# 3. El ecosistema Python de control: madurez real de las alternativas

Antes de construir algo propio, corresponde un mapa honesto de lo que ya existe:

| Librería | Qué es | Madurez | Cuándo usarla |
|---|---|---|---|
| **`python-control`** | El equivalente comunitario del Control System Toolbox: `tf`, `ss`, `bode`, `nyquist`, `rlocus`, `lqr`, `place`, MIMO, interconexiones, `margin`, `step_info`. | **Alta.** Proyecto activo desde ~2010, con suite de tests extensa. Es la referencia del área en Python. | Trabajo profesional o académico serio sobre LTI. |
| **`slycot`** | *Bindings* de Python a **SLICOT** (Fortran). `python-control` lo usa como backend opcional para los problemas numéricamente delicados (Riccati grandes, normas H∞, reducción de modelos). | Alta en lo numérico (es Fortran probado por décadas), pero su instalación requiere compilador Fortran y puede ser frágil. | Cuando `python-control` lo pida para un algoritmo avanzado. |
| **`scipy.signal` + `scipy.linalg`** | No es una librería "de control", pero contiene los ladrillos: `TransferFunction`, `lsim`, `cont2discrete`, `place_poles`, `solve_continuous_are`, `solve_lyapunov`. | **Muy alta** (es SciPy). | Como base numérica de cualquier desarrollo propio. **Es la que usamos acá.** |
| **`harold`** | Librería alternativa de control, con énfasis en representaciones polinómicas robustas y transformaciones de modelos. | Media; interesante pero con menos comunidad y mantenimiento intermitente. | Para estudiar enfoques alternativos de implementación. |
| **`sympy.physics.control`** | Control **simbólico** (exacto, sin coma flotante). | Media; útil pedagógicamente. | Derivaciones exactas y verificación de resultados a mano. |

## 3.1 ¿Entonces para qué un core propio?

Tres razones, en orden de honestidad:

1. **Pedagógica:** entender una herramienta exige poder leerla completa. `python-control` tiene decenas de miles de líneas; `lib_coreII` cabe en un cuaderno.
2. **Arquitectónica:** practicar las decisiones de diseño (representación, normalización, validación, API) que después permiten *leer críticamente* las librerías grandes.
3. **De soberanía técnica:** en entornos restringidos (un ESP32 con MicroPython, una RPi sin compilador Fortran, una notebook sin internet) saber reconstruir el 80 % útil con NumPy/SciPy puros tiene valor real.

La política correcta es: **usar `python-control` para producir, usar `lib_coreII` para comprender, y validar la segunda contra la primera** — que es exactamente lo que hace la sección 20.


# 4. Filosofía de diseño de `lib_coreII`

Mejoras estructurales respecto de las versiones anteriores (`lib_basica` y `lib_core`):

| Aspecto | Antes | Ahora |
|---|---|---|
| Márgenes | Primer cruce hallado sobre la grilla, sin refinar | **Todos** los cruces, interpolados en escala log, devolviendo el **peor caso** |
| Root locus | Nube de puntos (`np.roots` por cada K, sin orden) | **Ramas continuas** por emparejamiento de raíces entre pasos consecutivos |
| `step_info` | Valor final = última muestra; cruces sin interpolar | Valor final por promedio de cola, cruces 10–90 % interpolados, tiempo de pico y error estacionario |
| Construcción | Solo por coeficientes (`num`, `den`) | También **`TF.from_zpk`** (mejor condicionada para orden alto) |
| `minreal` | Cancelación que perdía la ganancia en algunos casos | Preserva la ganancia zpk exacta |
| Estabilidad | Solo raíces | Raíces **y** Routh–Hurwitz (con manejo de ceros en primera columna) |
| Retardo | No disponible | Aproximante de **Padé** implementado desde la fórmula |
| Síntesis | Compensadores manuales | Además: **diseño automático** de lead por margen de fase (Ogata) y de lag por constante de error; **PID con derivada filtrada** |
| Moderno | Controlabilidad/observabilidad | Además: `place` (Tits–Yang vía SciPy), **LQR** (Riccati/LAPACK) y **observador de Luenberger** |
| Confianza | Sin tests | **Suite de autotests contra valores analíticos** + validación cruzada opcional con `python-control` |

Principios que se mantienen: SISO primero, transparencia total del código, denominador siempre mónico, `dt=None` ⇒ continuo, y documentación de cada función en estilo **Doxygen** (`@brief`, `@param`, `@return`), legible tanto por humanos como por `doxygen`/`doxypypy` si algún día este código migra a un repositorio.


# 5. Entorno y dependencias

El core usa exclusivamente el *stack* científico estándar, preinstalado en Colab:

- **NumPy** — arrays, polinomios (`polymul`, `polyadd`, `roots`), autovalores. Núcleo en C, álgebra en BLAS/LAPACK.
- **SciPy** — `signal` (LTI, simulación, `cont2discrete`, `place_poles`) y `linalg` (Lyapunov, Riccati). También delega en LAPACK.
- **Matplotlib** — visualización.

Es decir: cada vez que esta librería "calcula en serio", el trabajo pesado lo hacen las mismas rutinas Fortran/C de la sección 2. Nuestra responsabilidad es la capa de arriba.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, linalg
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("NumPy :", np.__version__)
import scipy; print("SciPy :", scipy.__version__)

# 6. Utilidades numéricas

Funciones internas de saneamiento. Parecen triviales pero son la primera línea de defensa de la robustez:

- `_trim_poly` evita que coeficientes "casi cero" de alto grado inflen artificialmente el orden y degraden `np.roots`.
- `_normalize_tf` impone denominador **mónico**: todas las comparaciones y operaciones posteriores quedan bien escaladas.
- `_real_if_close` descarta partes imaginarias residuales al reconstruir polinomios desde raíces conjugadas.


In [ ]:
# ============================================================
# BLOQUE 0 - Utilidades numericas
# ============================================================
EPS = 1e-12   ## @brief Tolerancia numerica global del core.


def _as_poly(x):
    """!
    @brief   Convierte una entrada arbitraria en un polinomio 1D de coeficientes float.
    @param   x  Escalar, lista, tupla o ndarray con coeficientes en potencias decrecientes.
    @return  ndarray 1D de dtype float.
    """
    p = np.atleast_1d(np.asarray(x, dtype=float)).flatten()
    return p if p.size else np.array([0.0])


def _trim_poly(p, tol=EPS):
    """!
    @brief   Elimina ceros iniciales (coeficientes de mayor grado nulos) de un polinomio.
    @param   p    Polinomio como ndarray (potencias decrecientes).
    @param   tol  Umbral por debajo del cual un coeficiente se considera cero.
    @return  Polinomio recortado; si todos los coeficientes son nulos devuelve [0.0].
    """
    p = _as_poly(p)
    idx = np.flatnonzero(np.abs(p) > tol)
    return p[idx[0]:] if idx.size else np.array([0.0])


def _normalize_tf(num, den):
    """!
    @brief   Normaliza un par (num, den) a forma monica en el denominador.
    @details Recorta ceros de alto grado, valida denominador no nulo y divide
             ambos polinomios por den[0]. Esta normalizacion mejora el
             condicionamiento de np.roots y de las operaciones polinomicas.
    @param   num  Numerador (potencias decrecientes).
    @param   den  Denominador (potencias decrecientes).
    @return  Tupla (num, den) normalizada.
    @throws  ValueError si el denominador es identicamente nulo.
    """
    num, den = _trim_poly(num), _trim_poly(den)
    if den.size == 1 and abs(den[0]) < EPS:
        raise ValueError("Denominador identicamente nulo.")
    return num / den[0], den / den[0]


def _real_if_close(x, tol=1e-9):
    """!
    @brief   Descarta partes imaginarias residuales producto de redondeo.
    @param   x    Array posiblemente complejo.
    @param   tol  Tolerancia relativa para considerar nula la parte imaginaria.
    @return  Array real si las partes imaginarias son despreciables; el original si no.
    """
    x = np.asarray(x)
    if np.iscomplexobj(x) and np.allclose(x.imag, 0, atol=tol * (1 + np.abs(x).max(initial=0))):
        return x.real.copy()
    return x


def _stable(poles, discrete):
    """!
    @brief   Test de estabilidad asintotica segun el dominio.
    @param   poles     Array de polos.
    @param   discrete  True para plano z (|p|<1), False para plano s (Re{p}<0).
    @return  bool.
    """
    poles = np.asarray(poles)
    if poles.size == 0:
        return True
    return bool(np.all(np.abs(poles) < 1.0)) if discrete else bool(np.all(poles.real < 0.0))

# 7. Clase `TF`: función de transferencia

Una sola clase cubre el dominio $s$ (continuo, `dt=None`) y el dominio $z$ (discreto, `dt=T_s`):

$$
G(s)=\frac{N(s)}{D(s)} \qquad\text{o}\qquad G(z)=\frac{N(z)}{D(z)}
$$

Puntos de diseño:

- **`TF.from_zpk(zeros, polos, k)`**: para orden alto, expandir polinomios a mano acumula error; construir desde raíces con `np.poly` y ganancia explícita está mejor condicionado y es lo que hacen internamente las herramientas maduras.
- **`system_type()`**: cuenta polos en el origen ($s=0$) o en $z=1$, que determina los errores estacionarios clásicos ($e_{ss}$ ante escalón, rampa, parábola).
- **Horizonte automático**: `_t_auto()` estima $t_{end}\approx 7/|\sigma_{dom}|$ a partir del polo dominante, como hace `step` de MATLAB.
- **Barrido automático**: `_w_auto()` centra el barrido logarítmico dos décadas alrededor de las singularidades, y en discreto lo recorta a la frecuencia de Nyquist $\pi/T_s$.
- El **álgebra de bloques** (`*`, `+`, `-`, `/`) verifica compatibilidad de `dt` antes de operar.


In [ ]:
# ============================================================
# BLOQUE 1 - Clase TF
# ============================================================
class TF:
    """!
    @brief   Funcion de transferencia SISO, continua (s) o discreta (z).
    @details Internamente almacena polinomios normalizados (denominador monico).
             dt = None  -> sistema continuo.
             dt = Ts>0  -> sistema discreto con periodo de muestreo Ts.
    """

    def __init__(self, num, den, name="G", dt=None):
        """!
        @brief Construye G = num/den.
        @param num  Coeficientes del numerador (potencias decrecientes).
        @param den  Coeficientes del denominador (potencias decrecientes).
        @param name Etiqueta para reportes y graficos.
        @param dt   None (continuo) o periodo de muestreo (discreto).
        """
        self.num, self.den = _normalize_tf(num, den)
        self.name = name
        self.dt = dt

    # ---------- constructores alternativos ----------

    @classmethod
    def from_zpk(cls, zeros, poles, k=1.0, name="G", dt=None):
        """!
        @brief   Construye una TF desde ceros, polos y ganancia.
        @details Forma preferida cuando se conocen las raices: evita perdida de
                 precision al expandir polinomios de alto grado a mano.
        @param   zeros  Iterable de ceros.
        @param   poles  Iterable de polos.
        @param   k      Ganancia de la forma zpk.
        @return  Instancia TF con coeficientes reales (si las raices son conjugadas).
        """
        num = _real_if_close(k * np.poly(np.atleast_1d(zeros))) if len(np.atleast_1d(zeros)) else np.array([float(k)])
        den = _real_if_close(np.poly(np.atleast_1d(poles))) if len(np.atleast_1d(poles)) else np.array([1.0])
        return cls(num, den, name=name, dt=dt)

    # ---------- propiedades ----------

    @property
    def is_discrete(self):
        """! @brief True si el sistema es discreto (dt definido). """
        return self.dt is not None

    @property
    def order(self):
        """! @brief Orden del sistema = grado del denominador. """
        return len(self.den) - 1

    @property
    def relative_degree(self):
        """! @brief Grado relativo = grado(den) - grado(num). """
        return (len(self.den) - 1) - (len(self.num) - 1)

    def __repr__(self):
        var = "z" if self.is_discrete else "s"
        tag = f"  [dt = {self.dt} s]" if self.is_discrete else ""
        return f"{self.name}({var}) = ({np.poly1d(self.num)}) / ({np.poly1d(self.den)}){tag}"

    # ---------- interoperabilidad ----------

    def scipy(self):
        """!
        @brief   Devuelve el objeto LTI equivalente de scipy.signal.
        @return  signal.TransferFunction (continuo) o signal.dlti (discreto).
        """
        if self.is_discrete:
            return signal.dlti(self.num, self.den, dt=self.dt)
        return signal.TransferFunction(self.num, self.den)

    def to_ss(self, name=None):
        """!
        @brief   Conversion a espacio de estados (forma canonica de controlador).
        @return  Instancia SS equivalente.
        """
        A, B, C, D = signal.tf2ss(self.num, self.den)
        return SS(A, B, C, D, name=name or self.name + "_ss", dt=self.dt)

    # ---------- analisis algebraico ----------

    def poles(self):
        """! @brief Polos = raices del denominador. @return ndarray complejo. """
        return np.roots(self.den)

    def zeros(self):
        """! @brief Ceros finitos = raices del numerador. @return ndarray complejo. """
        if len(self.num) == 1:
            return np.array([])
        return np.roots(self.num)

    def dc_gain(self):
        """!
        @brief   Ganancia estatica: G(0) en continuo, G(1) en discreto.
        @return  float o inf si hay integradores puros.
        """
        x0 = 1.0 if self.is_discrete else 0.0
        d = np.polyval(self.den, x0)
        return np.inf if abs(d) < EPS else float(np.polyval(self.num, x0) / d)

    def system_type(self):
        """!
        @brief   Tipo del sistema = cantidad de polos en el origen (s=0) o en z=1.
        @return  Entero >= 0. Determina los errores estacionarios clasicos.
        """
        target = 1.0 if self.is_discrete else 0.0
        return int(np.sum(np.abs(self.poles() - target) < 1e-7))

    def is_stable(self):
        """! @brief Estabilidad asintotica de los polos. @return bool. """
        return _stable(self.poles(), self.is_discrete)

    def eval(self, x):
        """!
        @brief   Evalua G en puntos arbitrarios del plano complejo.
        @param   x  Escalar o array (s o z segun dominio).
        @return  Valores complejos G(x).
        """
        return np.polyval(self.num, x) / np.polyval(self.den, x)

    def minreal(self, tol=1e-7):
        """!
        @brief   Cancelacion aproximada polo-cero (realizacion minima).
        @details Empareja cada cero con el polo mas cercano no usado; si la
                 distancia es menor que tol, ambos se eliminan. La ganancia zpk
                 se preserva exactamente.
        @param   tol  Distancia maxima para considerar cancelacion.
        @return  Nueva TF reducida.
        """
        z, p = list(self.zeros()), list(self.poles())
        k = self.num[0]  # den es monico -> k de zpk = coef. lider del num
        used, z_keep = set(), []
        for zi in z:
            d = [np.inf if j in used else abs(zi - pj) for j, pj in enumerate(p)]
            j = int(np.argmin(d)) if d else -1
            if j >= 0 and d[j] < tol:
                used.add(j)
            else:
                z_keep.append(zi)
        p_keep = [pj for j, pj in enumerate(p) if j not in used]
        return TF.from_zpk(z_keep, p_keep, k, name=self.name + "_min", dt=self.dt)

    # ---------- respuesta temporal ----------

    def _t_auto(self):
        """!
        @brief   Horizonte temporal automatico segun el polo dominante.
        @return  t_end sugerido (float), acotado para evitar horizontes absurdos.
        """
        p = self.poles()
        re = np.abs(p.real[np.abs(p.real) > 1e-6]) if p.size else np.array([])
        if re.size == 0:
            return 10.0
        return float(np.clip(7.0 / re.min(), 1e-3, 1e4))

    def step(self, t_end=None, n=2000):
        """!
        @brief   Respuesta al escalon unitario.
        @param   t_end  Horizonte; None -> estimado automaticamente.
        @param   n      Numero de muestras.
        @return  (t, y) como ndarrays.
        """
        if self.is_discrete:
            t, y = signal.dstep((self.num, self.den, self.dt), n=int(n))
            return np.squeeze(t), np.squeeze(y)
        t = np.linspace(0, t_end or self._t_auto(), n)
        return signal.step(self.scipy(), T=t)

    def impulse(self, t_end=None, n=2000):
        """!
        @brief   Respuesta al impulso unitario.
        @return  (t, y) como ndarrays.
        """
        if self.is_discrete:
            t, y = signal.dimpulse((self.num, self.den, self.dt), n=int(n))
            return np.squeeze(t), np.squeeze(y)
        t = np.linspace(0, t_end or self._t_auto(), n)
        return signal.impulse(self.scipy(), T=t)

    def lsim(self, u, t):
        """!
        @brief   Simulacion ante entrada arbitraria u(t).
        @param   u  Vector de entrada.
        @param   t  Vector de tiempo (uniforme).
        @return  (t, y).
        """
        if self.is_discrete:
            t_out, y = signal.dlsim((self.num, self.den, self.dt), u, t=t)
            return np.squeeze(t_out), np.squeeze(y)
        t_out, y, _ = signal.lsim(self.scipy(), U=u, T=t)
        return t_out, y

    # ---------- respuesta frecuencial ----------

    def freqresp(self, w=None):
        """!
        @brief   Respuesta en frecuencia H(jw) (o H(e^{jw dt}) en discreto).
        @param   w  Vector de frecuencias [rad/s]; None -> barrido automatico.
        @return  (w, H) con H complejo.
        """
        if w is None:
            w = self._w_auto()
        x = np.exp(1j * w * self.dt) if self.is_discrete else 1j * w
        return w, self.eval(x)

    def _w_auto(self, n=4000):
        """!
        @brief   Barrido logaritmico automatico centrado en las singularidades.
        @return  ndarray de frecuencias [rad/s].
        """
        feats = np.abs(np.concatenate([self.poles(), self.zeros()]))
        feats = feats[feats > 1e-6]
        lo, hi = (1e-2, 1e2) if feats.size == 0 else (feats.min() / 100, feats.max() * 100)
        if self.is_discrete:
            hi = min(hi, np.pi / self.dt * 0.999)
        return np.logspace(np.log10(lo), np.log10(hi), n)

    def bode(self, w=None):
        """!
        @brief   Datos de Bode: magnitud [dB] y fase desenrollada [grados].
        @return  (w, mag_dB, fase_deg).
        """
        w, H = self.freqresp(w)
        mag = 20 * np.log10(np.maximum(np.abs(H), EPS))
        phase = np.degrees(np.unwrap(np.angle(H)))
        return w, mag, phase

    # ---------- algebra de bloques ----------

    def _check_dt(self, other):
        if isinstance(other, TF) and self.dt != other.dt:
            raise ValueError("Operacion entre sistemas con dt distinto.")

    def __mul__(self, other):
        """! @brief Conexion en cascada (serie) o escalado por ganancia. """
        if isinstance(other, TF):
            self._check_dt(other)
            return TF(np.polymul(self.num, other.num), np.polymul(self.den, other.den),
                      name=f"({self.name}*{other.name})", dt=self.dt)
        return TF(self.num * float(other), self.den, name=f"{other}*{self.name}", dt=self.dt)

    __rmul__ = __mul__

    def __add__(self, other):
        """! @brief Conexion en paralelo o suma de constante. """
        if isinstance(other, TF):
            self._check_dt(other)
            num = np.polyadd(np.polymul(self.num, other.den), np.polymul(other.num, self.den))
            return TF(num, np.polymul(self.den, other.den), name=f"({self.name}+{other.name})", dt=self.dt)
        return TF(np.polyadd(self.num, float(other) * self.den), self.den,
                  name=f"({self.name}+{other})", dt=self.dt)

    __radd__ = __add__

    def __sub__(self, other):
        return self + (-1) * other

    def __neg__(self):
        return (-1) * self

    def __truediv__(self, other):
        """! @brief Division de bloques (inversion del segundo). """
        if isinstance(other, TF):
            self._check_dt(other)
            return TF(np.polymul(self.num, other.den), np.polymul(self.den, other.num),
                      name=f"({self.name}/{other.name})", dt=self.dt)
        return TF(self.num / float(other), self.den, name=f"({self.name}/{other})", dt=self.dt)

# 8. Clase `SS`: espacio de estados

$$
\dot{x}=Ax+Bu,\qquad y=Cx+Du
$$

Decisiones de robustez:

- Los polos se obtienen como **autovalores de $A$** (`eigvals` → LAPACK `dgeev`), numéricamente preferible a pasar por el polinomio característico.
- Los tests de Kalman usan **rango por SVD** (`matrix_rank`), el criterio numéricamente correcto: la eliminación gaussiana puede dar rango lleno espurio en matrices mal condicionadas.

$$
\mathcal{C}=\begin{bmatrix}B & AB & \cdots & A^{n-1}B\end{bmatrix},\qquad
\mathcal{O}=\begin{bmatrix}C \\ CA \\ \vdots \\ CA^{n-1}\end{bmatrix}
$$

- Los **gramianos** resuelven Lyapunov con `scipy.linalg` en lugar de integrar numéricamente:

$$
AW_c+W_cA^T+BB^T=0,\qquad A^TW_o+W_oA+C^TC=0
$$


In [ ]:
# ============================================================
# BLOQUE 2 - Clase SS
# ============================================================
class SS:
    """!
    @brief   Modelo en espacio de estados: xdot = Ax + Bu ; y = Cx + Du.
    @details Soporta sistemas continuos y discretos (dt). El analisis de rango
             usa SVD (numpy.linalg.matrix_rank), criterio numericamente mas
             robusto que la eliminacion gaussiana.
    """

    def __init__(self, A, B, C, D=None, name="SS", dt=None):
        """!
        @brief Construye el modelo y valida dimensiones.
        @param A Matriz de estados (n x n).
        @param B Matriz de entrada (n x m).
        @param C Matriz de salida (p x n).
        @param D Matriz de transmision directa (p x m); None -> ceros.
        """
        self.A = np.atleast_2d(np.asarray(A, dtype=float))
        self.B = np.atleast_2d(np.asarray(B, dtype=float))
        self.C = np.atleast_2d(np.asarray(C, dtype=float))
        if self.B.shape[0] != self.A.shape[0] and self.B.shape[1] == self.A.shape[0]:
            self.B = self.B.T
        self.D = (np.zeros((self.C.shape[0], self.B.shape[1])) if D is None
                  else np.atleast_2d(np.asarray(D, dtype=float)))
        self.name, self.dt = name, dt
        self._validate()

    def _validate(self):
        """! @brief Chequeo de consistencia dimensional A/B/C/D. @throws ValueError. """
        n = self.A.shape[0]
        if self.A.shape != (n, n):
            raise ValueError("A debe ser cuadrada.")
        if self.B.shape[0] != n:
            raise ValueError("B: filas != n.")
        if self.C.shape[1] != n:
            raise ValueError("C: columnas != n.")
        if self.D.shape != (self.C.shape[0], self.B.shape[1]):
            raise ValueError("D: dimensiones p x m incorrectas.")

    @property
    def is_discrete(self):
        """! @brief True si dt esta definido. """
        return self.dt is not None

    @property
    def n_states(self):
        """! @brief Numero de estados n. """
        return self.A.shape[0]

    def __repr__(self):
        kind = "discreto" if self.is_discrete else "continuo"
        return f"{self.name}: SS {kind} | n={self.n_states}, m={self.B.shape[1]}, p={self.C.shape[0]}"

    def poles(self):
        """! @brief Polos = autovalores de A. """
        return np.linalg.eigvals(self.A)

    def is_stable(self):
        """! @brief Estabilidad asintotica via autovalores. """
        return _stable(self.poles(), self.is_discrete)

    def to_tf(self, input_index=0, output_index=0, name=None):
        """!
        @brief   Conversion a TF (canal SISO seleccionable en sistemas MIMO).
        @param   input_index   Indice de entrada.
        @param   output_index  Indice de salida.
        @return  Instancia TF.
        """
        num, den = signal.ss2tf(self.A, self.B, self.C, self.D, input=input_index)
        return TF(num[output_index], den, name=name or self.name + "_tf", dt=self.dt)

    # ---------- controlabilidad / observabilidad ----------

    def ctrb(self):
        """! @brief Matriz de controlabilidad [B AB ... A^{n-1}B]. """
        return np.hstack([np.linalg.matrix_power(self.A, i) @ self.B for i in range(self.n_states)])

    def obsv(self):
        """! @brief Matriz de observabilidad [C; CA; ...; CA^{n-1}]. """
        return np.vstack([self.C @ np.linalg.matrix_power(self.A, i) for i in range(self.n_states)])

    def is_controllable(self, tol=None):
        """! @brief Test de Kalman por rango (SVD). @return bool. """
        return np.linalg.matrix_rank(self.ctrb(), tol=tol) == self.n_states

    def is_observable(self, tol=None):
        """! @brief Test de Kalman dual por rango (SVD). @return bool. """
        return np.linalg.matrix_rank(self.obsv(), tol=tol) == self.n_states

    def gram(self, kind="c"):
        """!
        @brief   Gramiano de controlabilidad ('c') u observabilidad ('o').
        @details Resuelve la ecuacion de Lyapunov correspondiente con
                 scipy.linalg (rutinas LAPACK). Requiere sistema estable.
        @return  Matriz simetrica n x n.
        """
        if kind == "c":
            if self.is_discrete:
                return linalg.solve_discrete_lyapunov(self.A, self.B @ self.B.T)
            return linalg.solve_continuous_lyapunov(self.A, -(self.B @ self.B.T))
        if self.is_discrete:
            return linalg.solve_discrete_lyapunov(self.A.T, self.C.T @ self.C)
        return linalg.solve_continuous_lyapunov(self.A.T, -(self.C.T @ self.C))

    # ---------- simulacion / discretizacion ----------

    def discretize(self, dt, method="zoh", name=None):
        """!
        @brief   Discretizacion del modelo continuo.
        @param   dt      Periodo de muestreo.
        @param   method  'zoh', 'bilinear' (Tustin), 'euler', 'foh', etc.
        @return  Nueva SS discreta.
        """
        Ad, Bd, Cd, Dd, dtd = signal.cont2discrete((self.A, self.B, self.C, self.D), dt=dt, method=method)
        return SS(Ad, Bd, Cd, Dd, name=name or self.name + "_d", dt=dtd)

    def step(self, t_end=10, n=2000):
        """! @brief Respuesta al escalon via conversion interna a TF. """
        return self.to_tf().step(t_end=t_end, n=n)

    def lsim(self, u, t, x0=None):
        """!
        @brief   Simulacion ante entrada arbitraria con estado inicial.
        @param   u   Entrada (len(t) x m) o vector.
        @param   t   Vector de tiempo.
        @param   x0  Estado inicial; None -> origen.
        @return  (t, y, x).
        """
        if self.is_discrete:
            t_out, y, x = signal.dlsim((self.A, self.B, self.C, self.D, self.dt), u, t=t, x0=x0)
        else:
            t_out, y, x = signal.lsim((self.A, self.B, self.C, self.D), U=u, T=t, X0=x0)
        return t_out, np.squeeze(y), x

# 9. Interconexiones

Para realimentación con $G=N/D$ y $H$ en el lazo:

$$
T(s)=\frac{G}{1+GH}\;\Rightarrow\; D_{cl}=D_{GH}+N_{GH},\quad N_{cl}=N_G\,D_H
$$

`series` y `parallel` aceptan N bloques. Todo verifica consistencia de `dt`.


In [ ]:
# ============================================================
# BLOQUE 3 - Interconexiones
# ============================================================
def series(*systems, name=None):
    """!
    @brief   Conexion en cascada de N bloques TF.
    @return  TF producto.
    """
    out = systems[0]
    for s in systems[1:]:
        out = out * s
    if name:
        out.name = name
    return out


def parallel(*systems, name=None):
    """! @brief Conexion en paralelo (suma) de N bloques TF. @return TF suma. """
    out = systems[0]
    for s in systems[1:]:
        out = out + s
    if name:
        out.name = name
    return out


def feedback(G, H=None, sign=-1, name="T"):
    """!
    @brief   Lazo de realimentacion T = G / (1 -+ sign * G*H).
    @details Para realimentacion negativa unitaria: T = G/(1+G).
             Calculo polinomico directo: si GH = N/D, den_cl = D + N (sign=-1).
    @param   G     Bloque en la rama directa (TF).
    @param   H     Bloque en la rama de realimentacion; None -> unitaria.
    @param   sign  -1 negativa (default), +1 positiva.
    @return  TF de lazo cerrado.
    """
    if not isinstance(G, TF):
        raise TypeError("feedback opera sobre TF SISO.")
    H = H if H is not None else TF([1], [1], name="1", dt=G.dt)
    if G.dt != H.dt:
        raise ValueError("G y H deben compartir dt.")
    GH = G * H
    den_cl = np.polyadd(GH.den, GH.num) if sign == -1 else np.polysub(GH.den, GH.num)
    num_cl = np.polymul(G.num, H.den)
    return TF(num_cl, den_cl, name=name, dt=G.dt)

# 10. Métricas temporales: `step_info`

Mejoras concretas frente a la versión anterior:

- **Valor final** = promedio del 2 % final de la traza (una sola muestra final es sensible al ruido de integración).
- **Tiempo de subida 10–90 %** con **interpolación lineal** entre muestras (no el índice crudo).
- **Tiempo de pico** y **error estacionario** respecto de la referencia, ausentes antes.
- Manejo de respuestas de valor final negativo.

Recordatorio teórico para validar (2.º orden subamortiguado):

$$
M_p = e^{-\pi\zeta/\sqrt{1-\zeta^2}}\cdot 100\%,\qquad t_p=\frac{\pi}{\omega_n\sqrt{1-\zeta^2}}
$$


In [ ]:
# ============================================================
# BLOQUE 4 - Metricas temporales
# ============================================================
def step_info(t, y, tolerance=0.02, reference=1.0):
    """!
    @brief   Metricas clasicas de la respuesta al escalon.
    @details Estima valor final por promedio del 2%% final de la traza (mas
             robusto que tomar la ultima muestra), tiempo de subida 10-90%%
             con interpolacion lineal, tiempo de pico, sobrepico porcentual,
             tiempo de establecimiento (ultima salida de la banda) y error
             estacionario respecto de la referencia.
    @param   t          Vector de tiempo.
    @param   y          Vector de salida.
    @param   tolerance  Banda de establecimiento (0.02 = 2%%).
    @param   reference  Valor de la referencia escalon.
    @return  dict con: y_final, y_pico, t_pico, sobrepico_pct,
             t_subida_10_90, t_establecimiento, error_estacionario.
    """
    t, y = np.asarray(t, float).flatten(), np.asarray(y, float).flatten()
    n_tail = max(1, int(0.02 * len(y)))
    y_final = float(np.mean(y[-n_tail:]))
    i_peak = int(np.argmax(np.abs(y)))
    y_peak, t_peak = float(y[i_peak]), float(t[i_peak])
    Mp = max(0.0, (abs(y_peak) - abs(y_final)) / abs(y_final) * 100) if abs(y_final) > EPS else np.nan

    def _cross(level):
        s = np.sign(y_final) or 1.0
        idx = np.flatnonzero(s * y >= s * level)
        if idx.size == 0:
            return np.nan
        i = idx[0]
        if i == 0:
            return float(t[0])
        return float(np.interp(level, [y[i - 1], y[i]], [t[i - 1], t[i]]))

    t10, t90 = _cross(0.1 * y_final), _cross(0.9 * y_final)
    tr = t90 - t10 if np.isfinite(t10) and np.isfinite(t90) else np.nan
    band = tolerance * max(abs(y_final), EPS)
    outside = np.flatnonzero(np.abs(y - y_final) > band)
    ts = float(t[min(outside[-1] + 1, len(t) - 1)]) if outside.size else float(t[0])
    return {"y_final": y_final, "y_pico": y_peak, "t_pico": t_peak,
            "sobrepico_pct": float(Mp), "t_subida_10_90": float(tr),
            "t_establecimiento": ts, "error_estacionario": float(reference - y_final)}

# 11. Márgenes de estabilidad robustos

La versión anterior tomaba el **primer** cruce sobre la grilla y leía el valor en esa muestra. Eso falla en dos casos reales:

1. **Cruces múltiples** (sistemas con resonancias o retardo): el margen que importa es el **peor caso**, no el primero.
2. **Grilla gruesa**: el cruce real cae entre dos muestras; sin interpolar, el error en $\omega_{cg}$ se traslada directo al margen de fase.

`margins` ahora detecta **todos** los cruces de $|G|=1$ y de $\angle G=-180°$, los refina por interpolación en $\log\omega$, y devuelve el mínimo PM y el mínimo GM:

$$
PM = 180° + \angle G(j\omega_{cg}), \qquad GM = \frac{1}{|G(j\omega_{cp})|}
$$


In [ ]:
# ============================================================
# BLOQUE 5 - Analisis frecuencial: margenes
# ============================================================
def margins(G, w=None):
    """!
    @brief   Margenes de ganancia y fase del lazo abierto G.
    @details Busca TODOS los cruces de |G|=1 (0 dB) y de fase = -180 grados,
             refinandolos por interpolacion lineal en escala logaritmica de
             frecuencia. Devuelve el caso mas desfavorable (menor PM, menor GM),
             que es el criterio correcto cuando hay cruces multiples.
    @param   G  TF de lazo abierto.
    @param   w  Barrido de frecuencias; None -> automatico denso.
    @return  dict: PM_deg, wcg_rad_s (cruce de ganancia), GM, GM_dB,
             wcp_rad_s (cruce de fase). Campos None si no hay cruce.
    """
    if w is None:
        w = G._w_auto(n=20000)
    w, H = G.freqresp(w)
    mag_db = 20 * np.log10(np.maximum(np.abs(H), EPS))
    phase = np.degrees(np.unwrap(np.angle(H)))
    logw = np.log10(w)

    def _crossings(curve, level):
        out = []
        d = curve - level
        idx = np.flatnonzero(np.diff(np.sign(d)) != 0)
        for i in idx:
            lw = np.interp(0.0, [d[i], d[i + 1]], [logw[i], logw[i + 1]])
            out.append(10 ** lw)
        return out

    PM = wcg = None
    for wc in _crossings(mag_db, 0.0):
        ph = float(np.interp(np.log10(wc), logw, phase))
        pm = 180.0 + ph
        if PM is None or pm < PM:
            PM, wcg = pm, wc

    GM = GM_dB = wcp = None
    for wc in _crossings(phase, -180.0):
        m = float(np.interp(np.log10(wc), logw, mag_db))
        gm_db = -m
        if GM_dB is None or gm_db < GM_dB:
            GM_dB, wcp = gm_db, wc
            GM = 10 ** (gm_db / 20)
    return {"PM_deg": PM, "wcg_rad_s": wcg, "GM": GM, "GM_dB": GM_dB, "wcp_rad_s": wcp}

# 12. Gráficos

`plot_bode` dibuja magnitud y fase en una sola figura con ejes compartidos y **anota los márgenes** sobre el gráfico (líneas verticales en $\omega_{cg}$ y $\omega_{cp}$), al estilo `margin()` de MATLAB. `plot_nyquist` marca el punto crítico $-1+0j$ y la rama simétrica.


In [ ]:
# ============================================================
# BLOQUE 6 - Graficos
# ============================================================
def plot_step(sys, t_end=None, n=2000, info=True):
    """!
    @brief   Grafica la respuesta al escalon y anota metricas.
    @param   sys   TF o SS.
    @param   info  Si True imprime step_info.
    @return  (t, y).
    """
    t, y = sys.step(t_end=t_end, n=n) if isinstance(sys, TF) else sys.step(t_end=t_end or 10, n=n)
    plt.figure()
    plt.plot(t, y, label=sys.name)
    plt.axhline(y[-1], ls="--", lw=1, color="gray")
    plt.title(f"Respuesta al escalon - {sys.name}")
    plt.xlabel("Tiempo [s]"); plt.ylabel("Salida"); plt.legend(); plt.show()
    if info:
        for k, v in step_info(t, y).items():
            print(f"  {k:>22s} : {v:.6g}")
    return t, y


def plot_impulse(sys, t_end=None, n=2000):
    """! @brief Grafica la respuesta al impulso. @return (t, y). """
    t, y = sys.impulse(t_end=t_end, n=n)
    plt.figure(); plt.plot(t, y, label=sys.name)
    plt.title(f"Respuesta al impulso - {sys.name}")
    plt.xlabel("Tiempo [s]"); plt.ylabel("Salida"); plt.legend(); plt.show()
    return t, y


def plot_bode(G, w=None, show_margins=True):
    """!
    @brief   Diagrama de Bode (magnitud + fase) con margenes anotados.
    @param   show_margins  Marca wcg/wcp y lineas de 0 dB y -180 grados.
    @return  (w, mag_dB, fase_deg).
    """
    w, mag, phase = G.bode(w)
    m = margins(G) if show_margins else {}
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
    ax1.semilogx(w, mag); ax1.axhline(0, ls="--", lw=1, color="gray")
    ax1.set_ylabel("Magnitud [dB]"); ax1.set_title(f"Bode - {G.name}")
    ax2.semilogx(w, phase); ax2.axhline(-180, ls="--", lw=1, color="gray")
    ax2.set_ylabel("Fase [grados]"); ax2.set_xlabel("Frecuencia angular [rad/s]")
    if m.get("wcg_rad_s"):
        ax1.axvline(m["wcg_rad_s"], ls=":", color="tab:green")
        ax2.axvline(m["wcg_rad_s"], ls=":", color="tab:green",
                    label=f"PM = {m['PM_deg']:.1f} deg @ {m['wcg_rad_s']:.3g} rad/s")
    if m.get("wcp_rad_s"):
        ax1.axvline(m["wcp_rad_s"], ls=":", color="tab:red",
                    label=f"GM = {m['GM_dB']:.1f} dB @ {m['wcp_rad_s']:.3g} rad/s")
        ax2.axvline(m["wcp_rad_s"], ls=":", color="tab:red")
    for ax in (ax1, ax2):
        if ax.get_legend_handles_labels()[0]:
            ax.legend()
    plt.tight_layout(); plt.show()
    return w, mag, phase


def plot_nyquist(G, w=None):
    """!
    @brief   Diagrama de Nyquist con rama simetrica y punto critico -1+0j.
    @return  H(jw) sobre la rama positiva.
    """
    if w is None:
        w = G._w_auto(n=6000)
    w, H = G.freqresp(w)
    plt.figure()
    plt.plot(H.real, H.imag, label="w > 0")
    plt.plot(H.real, -H.imag, "--", lw=1, label="w < 0 (simetria)")
    plt.scatter([-1], [0], marker="x", s=120, color="red", label="-1 + 0j")
    plt.axhline(0, lw=1); plt.axvline(0, lw=1)
    plt.title(f"Nyquist - {G.name}"); plt.xlabel("Re"); plt.ylabel("Im")
    plt.axis("equal"); plt.legend(); plt.show()
    return H


def root_locus(G, k_values=None, xlim=None, ylim=None, plot=True):
    """!
    @brief   Lugar geometrico de las raices para K en [k_min, k_max].
    @details Resuelve D(s) + K N(s) = 0 para cada K y EMPAREJA cada raiz con
             la mas cercana del paso anterior (asignacion greedy), de modo que
             las ramas se dibujan como curvas continuas y no como nube de
             puntos. Esto reproduce el aspecto del rlocus de MATLAB/Octave.
    @param   k_values  Vector de ganancias; None -> logspace(-3, 4).
    @return  (k_values, roots) con roots de forma (len(K), orden).
    """
    if k_values is None:
        k_values = np.logspace(-3, 4, 1500)
    n_branch = max(len(G.den), len(G.num)) - 1
    roots = np.full((len(k_values), n_branch), np.nan, dtype=complex)
    prev = None
    for i, K in enumerate(k_values):
        r = np.roots(np.polyadd(G.den, K * G.num))
        if prev is not None and len(r) == len(prev):
            ordered, pool = [], list(r)
            for rp in prev:
                j = int(np.argmin([abs(rp - rr) for rr in pool]))
                ordered.append(pool.pop(j))
            r = np.array(ordered)
        roots[i, :len(r)] = r
        prev = r
    if plot:
        plt.figure()
        for b in range(roots.shape[1]):
            plt.plot(roots[:, b].real, roots[:, b].imag, lw=1.5)
        p, z = G.poles(), G.zeros()
        if p.size:
            plt.scatter(p.real, p.imag, marker="x", s=120, color="k", label="Polos OL", zorder=5)
        if z.size:
            plt.scatter(z.real, z.imag, marker="o", facecolors="none",
                        edgecolors="k", s=120, label="Ceros OL", zorder=5)
        plt.axhline(0, lw=1); plt.axvline(0, lw=1)
        plt.title(f"Lugar geometrico de las raices - {G.name}")
        plt.xlabel("Re{s}"); plt.ylabel("Im{s}")
        if xlim: plt.xlim(xlim)
        if ylim: plt.ylim(ylim)
        if p.size or z.size:
            plt.legend()
        plt.show()
    return k_values, roots


def compare_step(systems, t_end=10, n=2000):
    """!
    @brief   Superpone respuestas al escalon de varios sistemas.
    @return  dict {nombre: step_info}.
    """
    plt.figure(); results = {}
    for s in systems:
        t, y = s.step(t_end=t_end, n=n)
        plt.plot(t, y, label=s.name)
        results[s.name] = step_info(t, y)
    plt.title("Comparacion de respuestas al escalon")
    plt.xlabel("Tiempo [s]"); plt.ylabel("Salida"); plt.legend(); plt.show()
    return results


def compare_bode(systems, w=None):
    """! @brief Superpone diagramas de Bode (magnitud y fase) de varios sistemas. """
    if w is None:
        w = np.logspace(-3, 3, 3000)
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(9, 7))
    for G in systems:
        wi, mag, phase = G.bode(w)
        ax1.semilogx(wi, mag, label=G.name)
        ax2.semilogx(wi, phase, label=G.name)
    ax1.axhline(0, ls="--", lw=1, color="gray"); ax2.axhline(-180, ls="--", lw=1, color="gray")
    ax1.set_ylabel("Magnitud [dB]"); ax1.set_title("Comparacion de Bode"); ax1.legend()
    ax2.set_ylabel("Fase [grados]"); ax2.set_xlabel("Frecuencia angular [rad/s]"); ax2.legend()
    plt.tight_layout(); plt.show()

# 13. Lugar geométrico de las raíces con ramas continuas

El locus resuelve, para cada $K$:

$$
D(s)+K\,N(s)=0
$$

El problema clásico de implementarlo "a mano": `np.roots` devuelve las raíces **en orden arbitrario** en cada paso, así que graficar columna contra columna produce una nube de puntos con saltos entre ramas.

La solución (la misma idea que usan las herramientas maduras): **emparejar** cada raíz del paso $K_{i}$ con la más cercana del paso $K_{i-1}$ (asignación greedy por distancia). Las ramas quedan como curvas continuas y el gráfico se vuelve legible e interpretable.


In [ ]:
# ============================================================
# BLOQUE 7 - Estabilidad algebraica: Routh-Hurwitz
# ============================================================
def routh_table(den, eps=1e-9, verbose=True):
    """!
    @brief   Tabla de Routh-Hurwitz del polinomio caracteristico.
    @details Maneja el caso de cero en la primera columna sustituyendolo por
             un epsilon pequeno (criterio clasico de perturbacion). Cuenta los
             cambios de signo de la primera columna = numero de raices en el
             semiplano derecho.
    @param   den      Polinomio caracteristico (potencias decrecientes).
    @param   eps      Valor de perturbacion para ceros en la primera columna.
    @param   verbose  Imprime la tabla.
    @return  (tabla ndarray, n_raices_SPD, estable bool).
    """
    a = _trim_poly(den)
    n = len(a) - 1
    cols = (n // 2) + 1
    T = np.zeros((n + 1, cols))
    T[0, :len(a[0::2])] = a[0::2]
    if n >= 1:
        T[1, :len(a[1::2])] = a[1::2]
    for i in range(2, n + 1):
        piv = T[i - 1, 0]
        if abs(piv) < eps:
            piv = eps
        for j in range(cols - 1):
            T[i, j] = (piv * T[i - 2, j + 1] - T[i - 2, 0] * T[i - 1, j + 1]) / piv
    first = T[:, 0].copy()
    first[np.abs(first) < eps] = eps
    sign_changes = int(np.sum(np.diff(np.sign(first)) != 0))
    if verbose:
        print("Tabla de Routh (primera columna decide):")
        for i, row in enumerate(T):
            print(f"  s^{n - i}: " + "  ".join(f"{v: .5g}" for v in row))
        print(f"Cambios de signo: {sign_changes} -> raices en SPD: {sign_changes}")
    return T, sign_changes, sign_changes == 0

# 14. Routh–Hurwitz

Criterio algebraico de estabilidad que **no requiere calcular raíces**: el número de cambios de signo en la primera columna de la tabla es el número de raíces en el semiplano derecho. Es especialmente valioso para:

- obtener **rangos simbólico-numéricos de ganancia estable** (barriendo $K$ sobre $D+KN$);
- verificar de forma independiente lo que dicen `np.roots` o los autovalores.

La implementación maneja el caso patológico de **cero en la primera columna** con la sustitución clásica por $\epsilon$.


In [ ]:
# ============================================================
# BLOQUE 8 - Discretizacion y retardo
# ============================================================
def discretize(G, dt, method="zoh", name=None):
    """!
    @brief   Discretiza una TF continua.
    @param   method  'zoh' (default), 'bilinear' (Tustin), 'euler', 'foh', etc.
    @return  TF discreta con el dt indicado.
    """
    numd, dend, dtd = signal.cont2discrete((G.num, G.den), dt=dt, method=method)
    return TF(np.squeeze(numd), np.squeeze(dend), name=name or G.name + f"_d({method})", dt=dtd)


def pade_delay(L, order=3, name=None):
    """!
    @brief   Aproximacion racional de Pade del retardo puro e^{-Ls}.
    @details Permite incluir retardos de transporte en el algebra polinomica
             (no representables exactamente como cociente de polinomios).
    @param   L      Retardo [s].
    @param   order  Orden n de la aproximacion (n,n) (3 suele ser suficiente).
    @return  TF aproximante del retardo.
    @note    Formula clasica del aproximante diagonal:
             c_k = (2n-k)! n! / [ (2n)! k! (n-k)! ];
             num = sum c_k (-Ls)^k ; den = sum c_k (+Ls)^k.
    """
    from math import factorial
    n = int(order)
    c = [factorial(2 * n - k) * factorial(n) / (factorial(2 * n) * factorial(k) * factorial(n - k))
         for k in range(n + 1)]
    num = np.array([c[k] * (-L) ** k for k in range(n, -1, -1)])
    den = np.array([c[k] * (+L) ** k for k in range(n, -1, -1)])
    return TF(num, den, name=name or f"e^(-{L}s)~Pade{n}")

# 15. Discretización y retardo de Padé

**Discretización** vía `scipy.signal.cont2discrete`: ZOH (exacta ante escalones mantenidos), Tustin/bilineal (preserva estabilidad, distorsiona frecuencia), Euler, FOH.

Verificación canónica: para $G(s)=\frac{1}{s+1}$ con ZOH, el polo discreto debe caer **exactamente** en $z=e^{-T_s}$ (es un test de la suite).

**Retardo de transporte** $e^{-Ls}$: no es racional, así que para incluirlo en el álgebra polinómica se usa el aproximante diagonal de **Padé** $(n,n)$, implementado desde la fórmula:

$$
e^{-Ls}\approx\frac{\sum_{k=0}^{n}c_k(-Ls)^k}{\sum_{k=0}^{n}c_k(Ls)^k},
\qquad c_k=\frac{(2n-k)!\,n!}{(2n)!\,k!\,(n-k)!}
$$

El aproximante tiene módulo 1 exacto (es pasa-todo) y reproduce la fase $-\omega L$ hasta frecuencias $\omega L \lesssim n$.


In [ ]:
# ============================================================
# BLOQUE 9 - Sintesis clasica
# ============================================================
def lead(z, p, K=1.0, name="C_lead"):
    """!
    @brief   Compensador de adelanto C(s) = K (s+z)/(s+p), con z < p.
    @return  TF del compensador.
    """
    if not z < p:
        print("Advertencia: en un adelanto se espera z < p.")
    return TF([K, K * z], [1, p], name=name)


def lag(z, p, K=1.0, name="C_lag"):
    """!
    @brief   Compensador de atraso C(s) = K (s+z)/(s+p), con p < z.
    @return  TF del compensador.
    """
    if not p < z:
        print("Advertencia: en un atraso se espera p < z.")
    return TF([K, K * z], [1, p], name=name)


def lead_lag(z_lead, p_lead, z_lag, p_lag, K=1.0, name="C_lead_lag"):
    """! @brief Compensador adelanto-atraso = K * lead * lag. @return TF. """
    C = K * lead(z_lead, p_lead) * lag(z_lag, p_lag)
    C.name = name
    return C


def pid(Kp=1.0, Ki=0.0, Kd=0.0, N=20.0, name="PID"):
    """!
    @brief   PID con derivada filtrada: C(s) = Kp + Ki/s + Kd s/(s/N' + 1).
    @details El filtro de la accion derivativa (polo en N' = N*wd) la hace
             propia y realizable, evitando amplificacion infinita de ruido.
    @param   N  Relacion de filtrado de la derivada (tipico 8-20).
    @return  TF del controlador.
    """
    C = TF([Kp], [1], name="P")
    if Ki:
        C = C + TF([Ki], [1, 0], name="I")
    if Kd:
        tau = 1.0 / max(N, EPS)
        C = C + TF([Kd, 0], [tau, 1], name="D")
    C.name = name
    return C


def design_lead_pm(G, PM_target_deg, safety_deg=5.0, name="C_lead_auto", verbose=True):
    """!
    @brief   Diseno analitico de adelanto por margen de fase (procedimiento de Ogata).
    @details Pasos:
             1) phi_max = PM_target - PM_actual + margen de seguridad;
             2) alpha = (1 - sin phi_max)/(1 + sin phi_max);
             3) nueva wcg = frecuencia donde |G| = -10 log10(1/alpha) dB;
             4) z = wm*sqrt(alpha), p = wm/sqrt(alpha), K = 1/alpha (forma estandar).
    @param   G               Planta (lazo abierto sin compensar).
    @param   PM_target_deg   Margen de fase deseado [grados].
    @param   safety_deg      Margen de seguridad por corrimiento de wcg.
    @return  TF del compensador de adelanto disenado.
    @throws  ValueError si phi_max > 75 grados (requeriria multiples etapas).
    """
    m0 = margins(G)
    if m0["PM_deg"] is None:
        raise ValueError("La planta no cruza 0 dB: ajustar ganancia antes del diseno.")
    phi_max = np.radians(PM_target_deg - m0["PM_deg"] + safety_deg)
    if phi_max <= 0:
        print("La planta ya cumple el margen pedido; se devuelve C(s)=1.")
        return TF([1], [1], name=name)
    if np.degrees(phi_max) > 75:
        raise ValueError("phi_max > 75 grados: usar dos etapas de adelanto.")
    alpha = (1 - np.sin(phi_max)) / (1 + np.sin(phi_max))
    w = G._w_auto(n=20000)
    w, H = G.freqresp(w)
    mag_db = 20 * np.log10(np.maximum(np.abs(H), EPS))
    target_db = -10 * np.log10(1 / alpha)
    d = mag_db - target_db
    idx = np.flatnonzero(np.diff(np.sign(d)) != 0)
    if idx.size == 0:
        raise ValueError("No se hallo la nueva frecuencia de cruce.")
    i = idx[-1]
    wm = 10 ** np.interp(0.0, [d[i + 1], d[i]], [np.log10(w[i + 1]), np.log10(w[i])])
    z, p = wm * np.sqrt(alpha), wm / np.sqrt(alpha)
    C = TF([1 / alpha, z / alpha], [1, p], name=name)  # K=1/alpha compensa atenuacion DC
    if verbose:
        print(f"PM inicial = {m0['PM_deg']:.2f} deg | phi_max = {np.degrees(phi_max):.2f} deg | "
              f"alpha = {alpha:.4f} | wm = {wm:.4g} rad/s | z = {z:.4g}, p = {p:.4g}")
        m1 = margins(C * G)
        print(f"PM obtenido = {m1['PM_deg']:.2f} deg @ {m1['wcg_rad_s']:.4g} rad/s")
    return C


def design_lag_ess(G, factor, decade_below=10.0, name="C_lag_auto", verbose=True):
    """!
    @brief   Diseno de atraso para multiplicar la ganancia de baja frecuencia.
    @details Aumenta la constante de error (Kp/Kv/Ka segun el tipo) por 'factor'
             sin mover apreciablemente la wcg: ubica el cero una decada por
             debajo de la wcg actual y el polo en z/factor.
    @param   G       Planta o lazo a compensar.
    @param   factor  Factor de mejora del error estacionario (beta > 1).
    @return  TF del compensador de atraso.
    """
    m0 = margins(G)
    wcg = m0["wcg_rad_s"] or 1.0
    z = wcg / decade_below
    p = z / factor
    C = TF([1, z], [1, p], name=name)
    if verbose:
        print(f"wcg = {wcg:.4g} rad/s -> z = {z:.4g}, p = {p:.4g} (beta = {factor})")
    return C

# 16. Síntesis clásica

Compensadores manuales (`lead`, `lag`, `lead_lag`), **PID con derivada filtrada** (la derivada pura $K_d s$ es impropia e irrealizable; el filtro $\frac{K_d s}{s/N+1}$ la hace causal y limita la amplificación de ruido), y dos **rutinas de diseño automático**:

## `design_lead_pm` — adelanto por margen de fase (procedimiento de Ogata)

1. Fase a aportar: $\phi_{max}=PM_{deseado}-PM_{actual}+\text{seguridad}$
2. $\alpha=\dfrac{1-\sin\phi_{max}}{1+\sin\phi_{max}}$
3. Nueva $\omega_{cg}$: donde $|G|_{dB}=-10\log_{10}(1/\alpha)$ (porque el lead aporta $+10\log(1/\alpha)$ dB en $\omega_m$)
4. $z=\omega_m\sqrt{\alpha},\quad p=\omega_m/\sqrt{\alpha},\quad K=1/\alpha$

## `design_lag_ess` — atraso por constante de error

Para multiplicar $K_p/K_v/K_a$ por un factor $\beta$ sin mover la $\omega_{cg}$: cero una década por debajo de $\omega_{cg}$, polo en $z/\beta$.


In [ ]:
# ============================================================
# BLOQUE 10 - Sintesis moderna (espacio de estados)
# ============================================================
def place(sys_ss, desired_poles):
    """!
    @brief   Ubicacion de polos por realimentacion de estados u = -Kx.
    @details Envuelve scipy.signal.place_poles (algoritmo de Tits-Yang /
             KNV, numericamente robusto frente a la formula de Ackermann).
    @param   sys_ss         Modelo SS controlable.
    @param   desired_poles  Polos deseados del lazo cerrado.
    @return  Matriz de ganancias K (m x n).
    """
    res = signal.place_poles(sys_ss.A, sys_ss.B, np.asarray(desired_poles))
    return res.gain_matrix


def lqr(sys_ss, Q, R):
    """!
    @brief   Regulador lineal cuadratico continuo.
    @details Resuelve la ecuacion algebraica de Riccati (scipy.linalg.
             solve_continuous_are, basada en LAPACK) y devuelve
             K = R^{-1} B^T P, junto con P y los autovalores de lazo cerrado.
    @param   Q  Peso de estados (n x n, semidefinida positiva).
    @param   R  Peso de control (m x m, definida positiva).
    @return  (K, P, autovalores_lazo_cerrado).
    """
    Q, R = np.atleast_2d(Q), np.atleast_2d(R)
    P = linalg.solve_continuous_are(sys_ss.A, sys_ss.B, Q, R)
    K = np.linalg.solve(R, sys_ss.B.T @ P)
    eig = np.linalg.eigvals(sys_ss.A - sys_ss.B @ K)
    return K, P, eig


def observer_gain(sys_ss, desired_poles):
    """!
    @brief   Ganancia L de un observador de Luenberger por dualidad.
    @details Ubica los polos de (A - L C) usando place sobre el sistema dual
             (A^T, C^T).
    @return  Matriz L (n x p).
    """
    res = signal.place_poles(sys_ss.A.T, sys_ss.C.T, np.asarray(desired_poles))
    return res.gain_matrix.T

# 17. Síntesis moderna (espacio de estados)

- **`place`**: ubicación de polos con $u=-Kx$, envolviendo `scipy.signal.place_poles` (algoritmo de Tits–Yang/KNV). Nota de robustez: la fórmula de Ackermann que se enseña en clase es numéricamente frágil para $n>5$; las herramientas serias no la usan.
- **`lqr`**: resuelve la ecuación algebraica de Riccati

$$
A^TP+PA-PBR^{-1}B^TP+Q=0,\qquad K=R^{-1}B^TP
$$

con `solve_continuous_are` (LAPACK). El LQR garantiza, en el caso nominal SISO, $PM \geq 60°$ y $GM=\infty$ — uno de los resultados más elegantes de la teoría.

- **`observer_gain`**: observador de Luenberger por **dualidad**: ubicar los polos de $(A-LC)$ equivale a hacer `place` sobre $(A^T, C^T)$.


In [ ]:
# ============================================================
# BLOQUE 11 - Modelos rapidos y reportes
# ============================================================
def first_order(tau, K=1.0, name="G1"):
    """! @brief Primer orden G = K/(tau s + 1). @return TF. """
    return TF([K], [tau, 1], name=name)


def second_order(wn, zeta, K=1.0, name="G2"):
    """! @brief Segundo orden canonico G = K wn^2/(s^2 + 2 zeta wn s + wn^2). @return TF. """
    return TF([K * wn**2], [1, 2 * zeta * wn, wn**2], name=name)


def integrator(K=1.0, name="I"):
    """! @brief Integrador puro K/s. @return TF. """
    return TF([K], [1, 0], name=name)


def report(sys):
    """!
    @brief   Reporte de consola: modelo, polos, ceros, ganancia, tipo y estabilidad.
    @param   sys  TF o SS.
    """
    print(sys)
    print("\n  Polos:", np.round(sys.poles(), 6))
    if isinstance(sys, TF):
        print("  Ceros:", np.round(sys.zeros(), 6))
        print(f"  Ganancia DC : {sys.dc_gain():.6g}")
        print(f"  Tipo        : {sys.system_type()}")
        print(f"  Grado rel.  : {sys.relative_degree}")
    print(f"  Estable     : {sys.is_stable()}\n")

# 18. Modelos rápidos y reportes

Constructores de plantas canónicas (`first_order`, `second_order`, `integrator`) y `report()` para inspección de consola: polos, ceros, ganancia DC, **tipo de sistema** y **grado relativo**.


In [ ]:
# ============================================================
# BLOQUE 12 - Suite de autotests
# ============================================================
def run_selftests(verbose=True):
    """!
    @brief   Suite minima de verificacion del core contra resultados analiticos.
    @details Cada test compara contra valores de la teoria (no contra otra
             libreria), de modo que valida la matematica y no solo la
             consistencia interna.
    @return  True si todos los tests pasan.
    """
    ok = True

    def check(name, cond):
        nonlocal ok
        ok = ok and bool(cond)
        if verbose:
            print(f"  [{'OK ' if cond else 'FALLA'}] {name}")

    # T1: lazo cerrado de G=1/(s(s+1)) -> T = 1/(s^2+s+1), wn=1, zeta=0.5
    G = TF([1], [1, 1, 0])
    T = feedback(G)
    check("lazo cerrado 1/(s(s+1))", np.allclose(T.den, [1, 1, 1]))

    # T2: dc gain y tipo
    check("dc_gain K/(tau s+1) = K", np.isclose(first_order(3, 4).dc_gain(), 4))
    check("tipo de 1/(s(s+2)) = 1", TF([1], [1, 2, 0]).system_type() == 1)

    # T3: estabilidad
    check("1/(s-1) inestable", not TF([1], [1, -1]).is_stable())
    check("1/(s+1) estable", TF([1], [1, 1]).is_stable())

    # T4: sobrepico teorico de 2do orden zeta=0.5 -> Mp = exp(-pi*zeta/sqrt(1-zeta^2)) ~ 16.3%
    G2 = second_order(wn=2, zeta=0.5)
    t, y = G2.step(t_end=12, n=8000)
    Mp = step_info(t, y)["sobrepico_pct"]
    check(f"Mp 2do orden zeta=0.5 ({Mp:.2f}% ~ 16.30%)", abs(Mp - 16.30) < 0.6)

    # T5: margen de fase de G=1/(s(s+1)): PM teorico ~ 51.83 deg en wcg ~ 0.786
    m = margins(TF([1], [1, 1, 0]))
    check(f"PM de 1/(s(s+1)) ({m['PM_deg']:.2f} ~ 51.83 deg)", abs(m["PM_deg"] - 51.827) < 0.2)
    check(f"wcg ({m['wcg_rad_s']:.4f} ~ 0.7862)", abs(m["wcg_rad_s"] - 0.7862) < 0.01)

    # T6: GM de G = 1/(s(s+1)(s+2)) -> cruce de fase en w=sqrt(2), |G| = 1/6, GM = 6 (15.56 dB)
    m = margins(TF([1], np.polymul([1, 1, 0], [1, 2])))
    check(f"GM de 1/(s(s+1)(s+2)) ({m['GM']:.3f} ~ 6)", abs(m["GM"] - 6) < 0.05)

    # T7: ida y vuelta TF <-> SS
    G3 = TF([2, 3], [1, 4, 5, 6])
    G3b = G3.to_ss().to_tf()
    check("TF -> SS -> TF preserva polos",
          np.allclose(np.sort_complex(G3.poles()), np.sort_complex(G3b.poles()), atol=1e-8))

    # T8: controlabilidad / observabilidad de forma canonica
    S = G3.to_ss()
    check("forma canonica controlable", S.is_controllable())
    check("observable", S.is_observable())

    # T9: Routh sobre s^3+s^2+2s+24 (inestable, 2 raices SPD)
    _, nspd, stable = routh_table([1, 1, 2, 24], verbose=False)
    check("Routh detecta 2 raices SPD", nspd == 2 and not stable)

    # T10: minreal cancela (s+1)/(s+1)(s+2) -> 1/(s+2)
    Gm = TF(np.polymul([1, 1], [1]), np.polymul([1, 1], [1, 2])).minreal()
    check("minreal cancela polo-cero", len(Gm.den) == 2 and np.isclose(Gm.den[-1], 2))

    # T11: LQR estabiliza doble integrador
    dint = SS([[0, 1], [0, 0]], [[0], [1]], [[1, 0]])
    K, P, eig = lqr(dint, np.eye(2), [[1]])
    check("LQR estabiliza doble integrador", np.all(eig.real < 0))

    # T12: place ubica polos pedidos
    Kp_ = place(dint, [-2, -3])
    eigp = np.linalg.eigvals(dint.A - dint.B @ Kp_)
    check("place ubica {-2,-3}", np.allclose(np.sort(eigp.real), [-3, -2], atol=1e-6))

    # T13: discretizacion ZOH de 1/(s+1) con dt=0.1 -> polo en e^{-0.1}
    Gd = discretize(TF([1], [1, 1]), dt=0.1)
    check("ZOH: polo z = e^{-dt}", np.isclose(Gd.poles()[0].real, np.exp(-0.1), atol=1e-9))

    # T14: design_lead_pm alcanza el margen pedido
    Gp = TF([4], [1, 2, 0])
    C = design_lead_pm(Gp, PM_target_deg=50, verbose=False)
    m = margins(C * Gp)
    check(f"lead automatico PM ({m['PM_deg']:.1f} ~ >=50)", m["PM_deg"] >= 49.0)

    print("\nResultado global:", "TODOS LOS TESTS PASARON" if ok else "HAY TESTS FALLIDOS")
    return ok

# 19. Suite de autotests contra resultados analíticos

Diferencia clave con un test de "consistencia interna": cada caso compara contra un **valor de la teoría**, calculado a mano. Si la matemática del core está mal, acá explota:

| Test | Valor analítico |
|---|---|
| $M_p$ de 2.º orden, $\zeta=0.5$ | $e^{-\pi\cdot 0.5/\sqrt{0.75}}=16.30\%$ |
| PM de $\frac{1}{s(s+1)}$ | $51.83°$ en $\omega_{cg}=0.786$ rad/s |
| GM de $\frac{1}{s(s+1)(s+2)}$ | cruce de fase en $\omega=\sqrt{2}$, $GM=6$ ($15.56$ dB) |
| ZOH de $\frac{1}{s+1}$, $T_s=0.1$ | polo en $z=e^{-0.1}$ exacto |
| Routh de $s^3+s^2+2s+24$ | 2 raíces en el SPD |


In [ ]:
# (tests)

In [ ]:
run_selftests()

# 20. Validación cruzada opcional contra `python-control`

Cerrando el círculo de la sección 3: además de validar contra la teoría, comparamos contra la librería de referencia del ecosistema. La celda instala `control` (solo en Colab; comentar si no hay red) y contrasta polos de lazo cerrado, márgenes y respuesta al escalón.


In [ ]:
# Ejecutar en Colab. Si no hay conexion, comentar esta linea.
%pip install control --quiet

In [ ]:
try:
    import control as ct

    G_ours = TF([1], [1, 3, 2, 0], name="G")
    G_ref = ct.tf([1], [1, 3, 2, 0])

    # 1) Polos de lazo cerrado
    p_ours = np.sort_complex(feedback(G_ours).poles())
    p_ref = np.sort_complex(ct.poles(ct.feedback(G_ref)))
    print("Polos CL (lib_coreII):", np.round(p_ours, 6))
    print("Polos CL (control)   :", np.round(p_ref, 6))
    print("Coinciden:", np.allclose(p_ours, p_ref, atol=1e-8), "\n")

    # 2) Margenes
    gm, pm, wcp, wcg = ct.margin(G_ref)
    m = margins(G_ours)
    print(f"PM  -> lib_coreII: {m['PM_deg']:.4f} deg | control: {pm:.4f} deg")
    print(f"GM  -> lib_coreII: {m['GM']:.4f}      | control: {gm:.4f}")
    print(f"wcg -> lib_coreII: {m['wcg_rad_s']:.4f}   | control: {wcg:.4f}")
    print(f"wcp -> lib_coreII: {m['wcp_rad_s']:.4f}   | control: {wcp:.4f}\n")

    # 3) Respuesta al escalon del lazo cerrado
    t = np.linspace(0, 20, 2000)
    _, y_ours = feedback(G_ours).step(t_end=20, n=2000)
    t_ref, y_ref = ct.step_response(ct.feedback(G_ref), T=t)
    err = np.max(np.abs(y_ours - y_ref))
    print(f"Error maximo entre respuestas al escalon: {err:.3e}")
    print("Validacion cruzada:", "APROBADA" if err < 1e-6 else "REVISAR")
except ImportError:
    print("python-control no esta instalado: se omite la validacion cruzada.")

# 21. Ejemplo 1: planta de primer orden

$$
G_1(s)=\frac{2}{5s+1} \qquad (\tau=5\text{ s},\;K=2)
$$

Verificación esperable: $t_{subida\,10\text{–}90}\approx 2.2\,\tau=11$ s; establecimiento al 2 % $\approx 4\tau=20$ s; sin sobrepico.


In [ ]:
G1 = first_order(tau=5, K=2, name="Primer orden")
report(G1)
t, y = plot_step(G1)
plot_bode(G1)

# 22. Ejemplo 2: segundo orden subamortiguado

$$
G_2(s)=\frac{\omega_n^2}{s^2+2\zeta\omega_n s+\omega_n^2},\qquad \omega_n=5,\;\zeta=0.25
$$

Predicción teórica: $M_p=e^{-\pi\zeta/\sqrt{1-\zeta^2}}=44.4\%$ y $t_p=\pi/(\omega_n\sqrt{1-\zeta^2})=0.649$ s. Comparar con lo que reporta `step_info`.


In [ ]:
G2 = second_order(wn=5, zeta=0.25, name="Segundo orden")
report(G2)
t, y = plot_step(G2)
Mp_teo = np.exp(-np.pi*0.25/np.sqrt(1-0.25**2))*100
tp_teo = np.pi/(5*np.sqrt(1-0.25**2))
print(f"\nTeoria -> Mp = {Mp_teo:.2f} %  |  t_pico = {tp_teo:.4f} s")
plot_bode(G2)

# 23. Ejemplo 3: lazo cerrado, Nyquist y lugar de raíces

$$
G(s)=\frac{1}{s(s+2)} \;\Rightarrow\; T(s)=\frac{1}{s^2+2s+1}
$$

El locus muestra las dos ramas partiendo de $s=0$ y $s=-2$, encontrándose en $s=-1$ y abriéndose verticales: con esta estructura, el lazo es estable para todo $K>0$ (verificable también con Routh).


In [ ]:
G = TF([1], [1, 2, 0], name="G")
T = feedback(G, name="T")
report(G); report(T)
plot_step(T)
plot_nyquist(G)
root_locus(G, xlim=(-6, 1), ylim=(-4, 4))
print(margins(G))

# 24. Ejemplo 4: Routh–Hurwitz y rango de ganancias estables

Para $G(s)=\dfrac{1}{s(s+1)(s+5)}$ con realimentación unitaria y ganancia $K$, el polinomio característico es

$$
s^3+6s^2+5s+K=0
$$

La condición de Routh da $0<K<30$. Verificamos la tabla en tres puntos.


In [ ]:
Gr = TF([1], np.polymul([1, 1, 0], [1, 5]), name="Gr")
for K in [10, 30, 50]:
    print(f"\n--- K = {K} ---")
    routh_table(np.polyadd(Gr.den, K * Gr.num))

# 25. Ejemplo 5: diseño automático de adelanto por margen de fase

Planta tipo 1:

$$
G_p(s)=\frac{4}{s(s+2)}
$$

Objetivo: $PM\geq 50°$. La rutina ejecuta el procedimiento de Ogata completo y reporta cada paso intermedio.


In [ ]:
Gp = TF([4], [1, 2, 0], name="Gp")
print("Margenes sin compensar:", margins(Gp), "\n")

C_lead = design_lead_pm(Gp, PM_target_deg=50, name="C_lead")
print("\n", C_lead, sep="")

OL = series(C_lead, Gp, name="C*Gp")
compare_bode([Gp, OL])
res = compare_step([feedback(Gp, name="CL sin compensar"),
                    feedback(OL, name="CL con adelanto")], t_end=8)
for nombre, info in res.items():
    print(f"\n{nombre}:")
    for k, v in info.items():
        print(f"  {k:>22s} : {v:.4g}")

# 26. Ejemplo 6: atraso para error estacionario

Sobre la misma planta, mejorar la constante de velocidad $K_v$ un factor $\beta=10$ casi sin tocar la dinámica de cruce:

$$
e_{ss,\text{rampa}}=\frac{1}{K_v},\qquad K_v=\lim_{s\to 0}sG_{OL}(s)
$$


In [ ]:
C_lag = design_lag_ess(Gp, factor=10, name="C_lag")
OL_lag = series(C_lag, Gp, name="C_lag*Gp")

# Kv = lim s->0 de s*G_OL(s): el minreal cancela el s comun antes de evaluar
Kv_antes = (TF([1, 0], [1]) * Gp).minreal().dc_gain()
Kv_despues = (TF([1, 0], [1]) * OL_lag).minreal().dc_gain()
print(f"Kv antes = {Kv_antes:.4g} -> e_ss rampa = {1/Kv_antes:.4g}")
print(f"Kv despues = {Kv_despues:.4g} -> e_ss rampa = {1/Kv_despues:.4g}")
print("\nMargenes antes :", margins(Gp))
print("Margenes despues:", margins(OL_lag))

# Verificacion temporal: respuesta a rampa
t = np.linspace(0, 30, 4000)
_, y0 = feedback(Gp).lsim(t, t)
_, y1 = feedback(OL_lag).lsim(t, t)
plt.figure()
plt.plot(t, t, "k--", lw=1, label="rampa de referencia")
plt.plot(t, y0, label="sin atraso")
plt.plot(t, y1, label="con atraso")
plt.title("Seguimiento de rampa"); plt.xlabel("Tiempo [s]"); plt.legend(); plt.show()

# 27. Ejemplo 7: retardo de transporte con Padé

Planta térmica típica con retardo:

$$
G(s)=\frac{e^{-0.5s}}{s+1}
$$

El retardo come fase linealmente con la frecuencia ($-\omega L$) sin tocar la magnitud: es el enemigo número uno del margen de fase en procesos industriales.


In [ ]:
D = pade_delay(L=0.5, order=3)
G_sin = TF([1], [1, 1], name="sin retardo")
G_con = series(G_sin, D, name="con retardo (Pade 3)")

compare_bode([G_sin, G_con], w=np.logspace(-2, 2, 3000))
print("Margenes sin retardo:", margins(2 * G_sin))
print("Margenes con retardo:", margins(2 * G_con))
compare_step([feedback(2 * G_sin, name="CL sin retardo"),
              feedback(2 * G_con, name="CL con retardo")], t_end=10)

# 28. Ejemplo 8: discretización ZOH vs. Tustin

$$
G(s)=\frac{25}{s^2+4s+25} \quad\xrightarrow{T_s=0.05\text{ s}}\quad G(z)
$$

ZOH es exacta para entradas mantenidas; Tustin mapea el eje $j\omega$ al círculo unitario preservando estabilidad pero distorsionando frecuencias (warping). Con $T_s$ chico frente a la dinámica, ambas convergen.


In [ ]:
Gc = second_order(wn=5, zeta=0.4, name="continua")
Gz1 = discretize(Gc, dt=0.05, method="zoh")
Gz2 = discretize(Gc, dt=0.05, method="bilinear")
print(Gz1); print(Gz2)
print("\nPolos continuos :", np.round(Gc.poles(), 4))
print("Polos ZOH       :", np.round(Gz1.poles(), 4), "(deben ser e^{p*Ts})")
print("e^{p*Ts} teorico:", np.round(np.exp(Gc.poles() * 0.05), 4))

tc, yc = Gc.step(t_end=3, n=2000)
t1, y1 = Gz1.step(n=60)
t2, y2 = Gz2.step(n=60)
plt.figure()
plt.plot(tc, yc, label="continua")
plt.step(t1, y1, where="post", label="ZOH")
plt.step(t2, y2, where="post", label="Tustin")
plt.title("Discretizacion ZOH vs Tustin (Ts = 0.05 s)")
plt.xlabel("Tiempo [s]"); plt.legend(); plt.show()

# 29. Ejemplo 9: espacio de estados, LQR y observador

Planta de tercer orden con integrador:

$$
G(s)=\frac{1}{s(s+1)(s+2)}
$$

Flujo completo de control moderno: conversión a SS, verificación de controlabilidad/observabilidad, **LQR** con pesos $Q=\mathrm{diag}(10,1,1)$, $R=1$, y **observador** con polos 4–5 veces más rápidos que los del regulador.


In [ ]:
P = TF([1], np.polymul([1, 1, 0], [1, 2]), name="P").to_ss(name="P_ss")
print(P)
print("Controlable:", P.is_controllable(), "| Observable:", P.is_observable())

K, Pric, eig_cl = lqr(P, Q=np.diag([10, 1, 1]), R=[[1]])
print("\nK (LQR) =", np.round(K, 4))
print("Polos de lazo cerrado:", np.round(eig_cl, 4))

# Lazo cerrado x_dot = (A - BK)x con condicion inicial
Acl = P.A - P.B @ K
cl = SS(Acl, P.B, P.C, name="LQR_CL")
t = np.linspace(0, 10, 1500)
_, y, x = cl.lsim(np.zeros_like(t), t, x0=[1, 0, 0])
plt.figure(); plt.plot(t, y, label="salida (regulacion desde x0)")
plt.title("Regulador LQR: respuesta a condicion inicial")
plt.xlabel("Tiempo [s]"); plt.legend(); plt.show()

# Observador 4-5x mas rapido que el polo dominante del regulador
dom = max(eig_cl.real)
L = observer_gain(P, desired_poles=5 * np.real(eig_cl) - [0, 0.1, 0.2])
print("L (observador) =", np.round(L.flatten(), 4))
print("Polos del observador:", np.round(np.linalg.eigvals(P.A - L @ P.C), 4))

# 30. Criterios prácticos de ingeniería

**Sistema lento** → polos dominantes cerca del origen, poco ancho de banda. Acciones: subir ganancia, adelanto, mover polos dominantes a la izquierda (place/LQR).

**Sistema oscilatorio** → $\zeta$ bajo, PM bajo, $M_p$ alto. Acciones: adelanto para inyectar fase en $\omega_{cg}$, bajar ganancia, o en SS: penalizar más los estados de velocidad en $Q$.

**Error estacionario grande** → revisar **tipo** del sistema y ganancia de baja frecuencia. Acciones: atraso, acción integral (con cuidado del *windup* en la implementación real), o aumentar $K$ si el margen lo tolera.

**Ruido amplificado** → ganancia de alta frecuencia excesiva, derivada sin filtrar. Acciones: limitar el cociente $p/z$ del adelanto (típico $\leq 10$), usar el PID con derivada filtrada de la sección 16, recortar ancho de banda.

**Retardo de transporte** → cada $\omega_{cg}\cdot L$ radianes de fase perdidos. Si $L$ es comparable al tiempo de respuesta deseado, los compensadores clásicos no alcanzan: pensar en predictor de Smith (en la hoja de ruta).


# 31. Hoja de ruta

Extensiones naturales, en orden sugerido de dificultad:

1. **Nichols** y sensibilidades $S$, $T$ (ya hay todo lo necesario: `freqresp` + álgebra).
2. **Predictor de Smith** para retardo dominante (composición de bloques existentes).
3. **LQG**: combinar `lqr` + filtro de Kalman (Riccati dual, `solve_continuous_are` sobre $(A^T, C^T)$).
4. **MIMO**: generalizar `feedback` a SS (fórmulas matriciales), valores singulares $\sigma(G(j\omega))$.
5. **Identificación**: mínimos cuadrados sobre datos de `lsim` reales (puente directo con el DAQ-S3 / ESP32: adquirir, identificar, diseñar, discretizar y volcar el controlador discreto al firmware).
6. **Exportación**: generar el código C del controlador discreto (ecuación en diferencias) listo para ESP-IDF.

El punto 5–6 es donde este core deja de ser un ejercicio y se vuelve cadena de herramientas completa: planta física → identificación → síntesis → `discretize` → firmware.


# 32. Referencias

1. K. Ogata, *Ingeniería de Control Moderna*, 5.ª ed., Pearson, 2010.
2. R. C. Dorf y R. H. Bishop, *Sistemas de Control Moderno*, 13.ª ed., Pearson, 2017.
3. G. F. Franklin, J. D. Powell y A. Emami-Naeini, *Feedback Control of Dynamic Systems*, 8.ª ed., Pearson, 2019.
4. C. Moler, "The Origins of MATLAB", MathWorks Technical Articles.
5. E. Anderson *et al.*, *LAPACK Users' Guide*, 3.ª ed., SIAM, 1999.
6. C. L. Lawson *et al.*, "Basic Linear Algebra Subprograms for Fortran Usage", *ACM TOMS*, vol. 5, n.º 3, 1979.
7. P. Benner *et al.*, "SLICOT — A Subroutine Library in Systems and Control Theory", *Applied and Computational Control, Signals, and Circuits*, 1999.
8. R. M. Murray *et al.*, *Python Control Systems Library*, https://python-control.readthedocs.io
9. P. Virtanen *et al.*, "SciPy 1.0: fundamental algorithms for scientific computing in Python", *Nature Methods*, vol. 17, 2020.

---

*lib_coreII — análisis y síntesis de plantas y controladores. Validada contra teoría (sección 19) y contra `python-control` (sección 20).*
